# 01.3 - Data Collection - Paperclip

[Paperclip](https://paperclip.gxl.ai) is a CLI and MCP tool for searching 150M+ scientific papers from PubMed Central, arXiv, medRxiv, bioRxiv, and OpenAlex. Commands compose statelessly (each result is stored server-side) enabling progressive search-refine-extract workflows.

## Notebook Setup

Install the Paperclip CLI, then run `paperclip login` once in a terminal to authenticate before using the cells below.

In [ ]:
!curl -fsSL https://paperclip.gxl.ai/install.sh | bash

In [ ]:
# Run once in a terminal to authenticate via OAuth before using this notebook
# !paperclip login

import dotenv
from genscai import paths

dotenv.load_dotenv(paths.root / "../.env")

## 01. Search

`search` uses hybrid BM25 + embedding retrieval and returns 1-2 sentence summaries per paper. Use `-n` to set the result count and `-s` to filter by source (`pmc`, `arxiv`, `medrxiv`, `biorxiv`).

In [ ]:
!paperclip search "infectious disease compartmental modeling" -n 20

In [ ]:
# Filter to medRxiv preprints only
!paperclip search "epidemic forecasting neural network" -n 10 -s medrxiv

## 02. Cat

`cat` retrieves the full text of a paper by DOI or paper ID. Use `--section` to retrieve a specific section (e.g. `Methods`, `Results`, `Abstract`).

In [ ]:
# Replace with a DOI from your search results above
DOI = "10.1101/2020.03.15.20036533"

!paperclip cat {DOI} --section Abstract

In [ ]:
!paperclip cat {DOI} --section Methods

## 03. Grep

`grep` runs a regex pattern across the full corpus and is optimized to run ~100x faster than a literal grep. Use it to find papers that contain specific terms, equations, or patterns.

In [ ]:
!paperclip grep "R_0.*basic reproduction number" -n 20

In [ ]:
!paperclip grep "SEIR.*COVID" -n 10

## 04. Stateful Exploration

Results are stored server-side. Pass `--from prev` to run the next command over the same result set rather than the full corpus, enabling progressive narrowing.

In [ ]:
# Step 1: broad search
!paperclip search "malaria transmission model" -n 30

In [ ]:
# Step 2: grep within those 30 papers
!paperclip grep "mosquito.*bite rate" --from prev

In [ ]:
# Step 3: further narrow to papers mentioning interventions
!paperclip grep "intervention|net|insecticide" --from prev

## 05. Map

`map` parallelizes an LLM query across all papers in the current result set. Instead of looping sequentially, it extracts structured answers from N papers in a single call.

In [ ]:
# Run a search first, then extract structured information from each paper in parallel
!paperclip search "vaccine efficacy infectious disease model" -n 15

In [ ]:
!paperclip map --from prev --query "What disease was modeled and what modeling framework was used?"

## 06. SQL

`sql` runs read-only queries against the paper metadata table. Useful for filtering by author, journal, year, or source before doing a semantic search.

In [ ]:
!paperclip sql "SELECT title, authors, year FROM papers WHERE source = 'medrxiv' AND year >= 2022 LIMIT 20"

In [ ]:
!paperclip sql "SELECT source, COUNT(*) as n FROM papers GROUP BY source ORDER BY n DESC"